# 🚀 Servidor OpenCode GPU (DeepSeek R1 + Qwen 2.5 Coder) - Google AI Pro

### ⚡ Instrucciones:
1. Verificá que arriba a la derecha esté conectada la **GPU T4**.
2. Ejecutá la **Celda 1** para iniciar el motor y descargar ambos modelos (**DeepSeek R1** y **Qwen 2.5 Coder**).
3. *(Opcional)*: Ejecutá la **Celda 2** si querés abrir una página web tipo ChatGPT para chatear directamente en el navegador.

In [ ]:
# 1. INICIAR MOTOR Y DESCARGAR DEEPSEEK R1 + QWEN 2.5 CODER
import os, time, re, subprocess

print("==================================================================")
print("1/4 📦 Instalando Ollama y Cloudflare Tunnel...")
print("==================================================================")
!curl -fsSL https://ollama.ai/install.sh | sh > /dev/null 2>&1
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

print("\n==================================================================")
print("2/4 ⚡ Iniciando motor Ollama en background...")
print("==================================================================")
!pkill -f "ollama serve" || true
!pkill -f "cloudflared" || true
time.sleep(2)

os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
os.environ["OLLAMA_ORIGINS"] = "*"
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(4)

print("\n==================================================================")
print("3/4 📥 Descargando Qwen 2.5 Coder y DeepSeek R1 en la GPU...")
print("==================================================================")
!ollama pull qwen2.5-coder:7b
!ollama pull deepseek-r1:8b

print("\n==================================================================")
print("4/4 🚀 Levantando túnel público de Cloudflare...")
print("==================================================================")
!rm -f tunnel.log
subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:11434"], stdout=open("tunnel.log", "w"), stderr=subprocess.STDOUT)

public_url = None
for _ in range(40):
    time.sleep(1)
    if os.path.exists("tunnel.log"):
        with open("tunnel.log", "r") as f:
            content = f.read()
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', content)
            if match:
                public_url = match.group(0)
                break

if public_url:
    print("\n" + "="*65)
    print("🟢 ¡TU SERVIDOR OPENCODE ESTÁ ACTIVO EN LA GPU DE GOOGLE!")
    print("="*65)
    print(f"\n👉 URL DEL TÚNEL PARA ANTIGRAVITY:")
    print(f"   {public_url}\n")
    print("👉 MODELOS DISPONIBLES: deepseek-r1:8b | qwen2.5-coder:7b")
    print("="*65)
    print("\n📋 Copiá la URL de arriba y pegala en el chat.")
    print("="*65 + "\n")
    try:
        while True:
            time.sleep(60)
    except KeyboardInterrupt:
        print("Túnel detenido.")
else:
    print("⚠️ No se encontró la URL en tunnel.log. Registros:")
    if os.path.exists("tunnel.log"):
        with open("tunnel.log", "r") as f:
            print(f.read())


In [ ]:
# 2. (OPCIONAL) INTERFAZ WEB TIPO CHATGPT EN TU NAVEGADOR
!pip install -q gradio
import gradio as gr
import requests, json

def chat(message, history, model, temp):
    msgs = []
    for u, a in history:
        msgs.append({"role": "user", "content": u})
        msgs.append({"role": "assistant", "content": a})
    msgs.append({"role": "user", "content": message})
    
    r = requests.post(
        "http://localhost:11434/api/chat",
        json={"model": model, "messages": msgs, "stream": True, "options": {"temperature": temp}},
        stream=True
    )
    acc = ""
    for line in r.iter_lines():
        if line:
            acc += json.loads(line).get("message", {}).get("content", "")
            yield acc

with gr.Blocks(theme=gr.themes.Soft(), title="Chefsy AI Studio") as demo:
    gr.Markdown("# 🤖 Chefsy AI Studio (Google Colab GPU)")
    gr.Markdown("Chatea directamente con **DeepSeek R1** y **Qwen 2.5 Coder** alojados en tu GPU T4.")
    with gr.Row():
        model_dropdown = gr.Dropdown(
            choices=["deepseek-r1:8b", "qwen2.5-coder:7b"],
            value="deepseek-r1:8b",
            label="Modelo de IA"
        )
        temp_slider = gr.Slider(0.0, 1.0, value=0.2, label="Temperatura / Creatividad")
    chat_ui = gr.ChatInterface(fn=chat, additional_inputs=[model_dropdown, temp_slider])

demo.launch(share=True)